# Bank Customer Churn — A Retention Decision

## The commercial problem

This dataset contains **10,000 customers** and a **20.4% churn rate**: **2,037 people left**, taking about **$186 million** in deposits with them. A customer who leaves does not only withdraw a balance. The bank also loses future fees, loan margin, and cross-sell. Replacing that relationship costs several times more than keeping it.

Churn here is quiet. There is no resignation letter. By the time an account has been dormant for months, the relationship is usually already gone. This notebook aims to move the bank from late-stage reviews to a ranked list of customers to contact while the relationship may still be saved.

## Why I chose this approach

I did not start with a model. A score is useful only after three business questions have been answered:

1. **Who is expensive to lose?** Age, balance, Germany, and product count answer that. Credit score, salary, and tenure do not.
2. **What can the bank actually change?** Activity and product policy can be changed. Age and country cannot, so they are used to prioritise, not as the campaign itself.
3. **How many people can the team contact?** Missing a churner is planned here at about **$5,000** of lifetime value. A wasted retention contact is about **$10**. The team cannot call the whole book, so the cutoff is a budget decision, not the default 0.5.

The notebook follows this business-first sequence:

| Step | Business reason |
|---|---|
| Check data quality | A retention list built on duplicates or blanks wastes the call-centre budget |
| Test every attribute against churn | Drop anything that does not change who we contact |
| Engineer only the patterns EDA proved | The product cliff and the Germany–balance pocket are not linear |
| Score models on F1, with SMOTE inside the training pipeline | "Everyone stays" is 80% accurate and saves nobody |
| Pick the contact threshold from cost and a 30% capacity cap | Precision and recall are translated into money and workload |

**Dataset:** [Bank Customer Churn — Kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction) · 10,000 customers · 14 fields · target `Exited` (1 = left, 0 = stayed).

**Notebook 1 of 4 — Data cleaning and EDA.**  Input: `data/Churn_Modelling.csv`. Next: `02_Feature_Engineering.ipynb`.

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "model"


## 1. Data loading

I drop `RowNumber`, `CustomerId`, and `Surname` before any analysis. They identify a person. They do not describe behaviour the bank can act on, and leaving them in would let a model memorise individuals instead of learning a pattern that works on the next customer.

In [ ]:
warnings.filterwarnings("ignore")
df =  pd.read_csv(DATA_DIR / "Churn_Modelling.csv")

df  = df.iloc[:,3:]

## 2. Data quality

A missing value or a duplicate row becomes a wrong name on a retention list. I check both before trusting any rate or any model.

In [ ]:
duplicate_count = df.duplicated().sum()

print(f"DUPLICATE ROWS : {duplicate_count}")


# Null Report
null_report = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing (%)":   df.isnull().mean().mul(100).round(2)
})

null_report = null_report[null_report["Missing Count"] > 0]

print(null_report if not null_report.empty else "No missing values")

## 3. How big is the problem?

About **one in five** customers leave. That is the number the rest of the notebook is built around.

A model that predicts "stay" for everyone is **80% accurate** and catches **zero** churners. Accuracy is the wrong scorecard. I care about finding the people who leave, without flooding the team with false alarms. Class imbalance is handled later, and only on the training data, so the test result stays honest.

In [ ]:
x = df["Exited"].value_counts()

plt.figure(figsize=(6,6))
plt.pie(x, 
        labels=["Stayed", "Left"], 
        autopct="%1.1f%%",
       colors=["yellow","red"],
        explode=(0.05, 0),
        startangle=90,
        )

plt.title("Churn Distribution", fontsize=14, fontweight="bold")
plt.show()



## 4. Gender

**Business question:** is the book one population, or do women and men leave at different rates?

I compare rates within each gender (a row-normalised crosstab), not raw counts. France-sized groups and Spain-sized groups should not be compared by headcount alone.

In [ ]:


# Create crosstab
ct = round(pd.crosstab(df["Gender"], df["Exited"], normalize="index") * 100, 2)

# Plot heatmap
plt.figure(figsize=(6,4))
sns.heatmap(ct, annot=True, fmt=".2f", cmap="Blues")
plt.title("Churn Rate by Gender (%)")
plt.xlabel("Exited (0=Stayed, 1=Churned)")
plt.ylabel("Gender")
plt.show()

## Finding: women leave more often — do not target offers by gender

Women churn at **25.1%**. Men churn at **16.5%**. That is about **1.5×**, an **8.6 point** gap.

**What this means for the bank**

- The gap is large enough that a single "average customer" campaign will under-serve part of the book.
- Gender is a **diagnostic**, not a cause, and not a fair or compliant way to price or select offers. The gap is likely sitting inside age, product mix, balance, or activity.

**Decision**

- Do not build a women-only offer. Build the contact list from things the bank can defend and change: inactivity, product count, balance, age band, and country.
- Before closing the gender question, check whether the gap remains inside those groups. If it does, treat it as a service issue (complaints, branch experience, product fit) and fix the service.

## 5. Credit score

**Business question:** should low credit score put someone on the retention list?

Credit score is the usual proxy for financial stress. If churners and stayers look the same, the bank should not spend relationship-manager time on credit bands.

In [ ]:
print(df.groupby("Exited")["CreditScore"].mean())

In [ ]:
sns.boxplot(data=df, x="Exited", y="CreditScore", palette="Blues")
plt.title("Credit Score by Churn Status")
plt.xticks([0,1], ["Stayed", "Churned"])
plt.show()


In [ ]:
cols = ["CreditScore", "Age", "Balance", "EstimatedSalary", "Tenure"]

for col in cols:
      churned_skew = round(df[df["Exited"] == 1][col].skew(), 2)
      stayed_skew  = round(df[df["Exited"] == 0][col].skew(), 2)
      print(f"{col:20} | Churned: {churned_skew:6} | Stayed: {stayed_skew:6}")


## 6. Age

**Business question:** is there an age band where the bank is losing a disproportionate share of relationships?

I test the working hypothesis that customers over 40 are the high-risk book, then check whether balance and tenure tell a different story. Age cannot be changed by a campaign. If it is the strongest split, it is used to **prioritise** who gets a relationship manager, not as the message of the campaign.

In [ ]:
round(df.groupby("Exited")["Age"].agg(["mean",  "std", "skew"]),2)   

In [ ]:
sns.boxenplot(data=df,x="Exited",y="Age",palette="Blues")
plt.title("Credit Score by Churn Status")
plt.xticks([0,1], ["Stayed", "Churned"])
plt.show()


### What the three distributions decide

- **Age is the split that matters.** People who left average **44.8 years**. People who stayed average **37.4**.
- **Balance is a value problem, not a poverty problem.** Leavers hold **more** money (about **$91k** vs **$73k**). Each exit is expensive.
- **Tenure is not a programme.** The two distributions sit on top of each other (about **5.0 vs 4.9 years**). A "loyalty year" campaign would not change who we call.

These three checks are why later models keep age and balance, and why tenure is not treated as a driver.

In [ ]:
cols = ["Age", "Balance", "Tenure"]

fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes = axes.flatten()

for i, col in enumerate(cols):
    sns.boxenplot(data=df, x="Exited", y=col, palette="coolwarm",hue="Exited", ax=axes[i])
    axes[i].set_title(f"{col} by Churn Status")
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(["Stayed", "Churned"])
    axes[i].set_xlabel("")

plt.suptitle("Numeric Features by Churn Status", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Active membership and credit card

**Business question:** which of these can a campaign actually move?

`IsActiveMember` is behaviour. The bank can change it with use of the app, a conversation, or a reason to log in. `HasCrCard` is a product flag. I check both so we do not build a card campaign on a difference that is not there.

In [ ]:


x = df["HasCrCard"].value_counts()

plt.figure(figsize=(6,6))
plt.pie(x, 
        labels=["Has Card", "No Card"], 
        autopct="%1.1f%%",
        colors=["green","lightblue"],
        explode=(0.05, 0),
        startangle=90,
       )

plt.title("Credit Card Distribution", fontsize=14, fontweight="bold")
plt.show()


In [ ]:
round(pd.crosstab(df["Exited"],df["HasCrCard"],normalize="columns")*100,2)

In [ ]:
ct = round(pd.crosstab(df["Exited"], df["IsActiveMember"], normalize="columns") * 100, 2)

ct.T.plot(kind="bar", figsize=(7,5), color=["lightblue","salmon"])
plt.title("Churn Rate by Active Member Status")
plt.xlabel("Is Active Member")
plt.xticks([0,1], ["Inactive","Active"], rotation=0)
plt.ylabel("Percentage %")
plt.legend(["Stayed","Churned"])
plt.tight_layout()
plt.show()

### Decision: inactivity is a campaign; the credit card is not

Inactive members churn at **26.9%**. Active members churn at **14.3%**. Inactive customers are about **64% of everyone who left** (1,302 of 2,037).

Credit-card holders and non-holders both churn at about **20%**. A card offer will not separate leavers from stayers.

**Why this is the lever I would fund first:** unlike age or country, activity can be changed this quarter. Re-activation (app prompt, relationship call, a reason to transact) is aimed at the inactive book, then ranked by balance so the calls land on the deposits that matter.

## 8. Balance and salary, split by gender

**Business question:** is the gender gap just "women hold different money," and is salary a targeting field?

If salary separates leavers from stayers, income bands belong on the contact list. If it does not, using salary would spend budget on a difference the data does not support.

In [ ]:
df.groupby(["Gender","Exited"])[["Balance","EstimatedSalary"]].mean().round(2).plot(
    kind="bar", figsize=(10,5), colormap="coolwarm")
plt.title("Average Financial Metrics by Gender and Churn")
plt.xticks(rotation=0)
plt.ylim(40000,110000)
plt.tight_layout()
plt.show()

## Finding: the bank is losing the higher balances, in both genders

Leavers hold more money than stayers, for women and for men. Salary sits near **$100k** in every group.

| Group | Avg. balance | Avg. salary |
|---|---|---|
| Female, stayed | ~$71k | ~$100k |
| Female, left | ~$89k | ~$103k |
| Male, stayed | ~$74k | ~$100k |
| Male, left | ~$94k | ~$99k |

**Decision**

- Put relationship-manager capacity on **high-balance** customers. A departure here costs more deposits than an average exit.
- Do not target by salary. Income does not explain who leaves.
- Do not explain the female gap as a balance story. Men and women who leave hold similar money, so the gender gap has to be checked inside age, products, and activity — and still must not become a gendered price.

## 9. Geography

**Business question:** is one market leaking twice as fast as the others, and is that market also where the deposits sit?

Country is not something a campaign changes. It decides **where** the retention team sits and whether the offer needs to be local.

In [ ]:
df.pivot_table(values=["Balance","CreditScore","Age","EstimatedSalary"],
               index="Geography",
               columns="Exited",
               aggfunc="mean").round(2)

In [ ]:

ct = pd.crosstab(df["Geography"], df["Exited"], normalize="index") * 100

sns.heatmap(ct, annot=True, fmt=".1f", cmap="viridis")
plt.title("Geography vs Gender Distribution (%)")
plt.show()

sns.countplot(data=df, x="Geography", hue="Exited", palette="coolwarm")
plt.title("Churn by Geography")
plt.xticks(rotation=0)
plt.legend(["Stayed","Churned"])
plt.show()

### Decision: Germany is a market problem, not a footnote

| Country | Churn | Customers | Avg. balance |
|---|---|---|---|
| France | 16.2% | 5,014 | ~$62k |
| Spain | 16.7% | 2,477 | ~$62k |
| Germany | **32.4%** | 2,509 | **~$120k** |

Germany is a quarter of the book and about **40% of all exits** (814 people). Those exits carried about **$98 million** in deposits — more than half of the **$186 million** that left the bank.

**Why a local programme, not a global email:** the same offer that is "fine" in France is being applied to a market with double the exit rate and double the balance. I would staff a Germany desk (local rate review, a named owner, German-language outreach) and track retained balance there separately. Headcount should follow the money, and the money is in Germany.

## 10. Number of products

**Business question:** does a deeper relationship protect the bank, the way cross-sell targets assume?

If more products meant more loyalty, the growth plan would be "sell a third product." I check the churn rate at 1, 2, 3, and 4 products before accepting that plan. A straight line cannot describe this if the risk jumps at a cliff.

In [ ]:
ct  = round(pd.crosstab(df["Exited"],df["NumOfProducts"],normalize="columns")*100,2)

# Assuming 'ct' is your crosstab result
ax = ct.T.plot(kind='bar', figsize=(10, 6), color=['#3498db', '#e74c3c'], edgecolor='black')

# 3. Add titles and labels
plt.title('Churn Rate % by Number of Products', fontsize=15, pad=20)
plt.xlabel('Number of Products', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.legend(title='Customer Status', labels=['Stayed (0)', 'Exited (1)'])
plt.xticks(rotation=0)  # Keeps the X-axis labels upright
plt.tight_layout()
plt.show()

## Decision: stop selling the 3rd and 4th product — do not spend the budget on 60 people

| Products | Customers | Churn | Exits | Deposits that left |
|---|---|---|---|---|
| 1 | 5,084 | 27.7% | 1,409 | **~$130 million** |
| 2 | 4,590 | **7.6%** | 348 | ~$31 million |
| 3 | 266 | 82.7% | 220 | (with 4-product exits) |
| 4 | 60 | **100%** | 60 | **~$25 million for 3 and 4 combined** |

A 100% churn rate looks like the top priority. It is not the P&L. Four-product customers are **60 people**. Three- and four-product exits together are **280 people and about $25 million**. One-product exits are **1,409 people and about $130 million**.

**What I would actually do**

1. **Change the product policy.** Two products is the stable relationship (7.6% churn). A third product is where loyalty breaks. Cap cross-sell at two unless a reviewer signs off.
2. **Put retention money on the one-product book**, especially where the balance is high. That is where the deposits already walked.
3. **Treat the 60 four-product customers as a case review**, not a campaign. The group is small enough to call every one of them and learn whether fees or overselling caused the exit. Do not size a bank-wide budget off a cell of 60.

### Why a linear model is the wrong tool here

| Products | Churn | Avg. balance |
|---|---|---|
| 1 | 27.7% | ~$98,600 |
| 2 | 7.6% | ~$51,900 |
| 3 | 82.7% | ~$75,500 |
| 4 | 100% | ~$93,700 |

Risk falls from one product to two, then collapses. Two-product customers also hold the **lowest** balances, so "loyal" and "high value" are not the same segment. That non-linear cliff is why I later use tree models and a `Products_Bucket` feature instead of hoping logistic regression draws a straight line through this.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Average balance for each product count
avg_balance = (df.groupby("NumOfProducts")["Balance"]
                 .mean()
                 .reset_index())

colors = ['red', '#3498db', 'red', 'red']  # highlight 3 and 4 (high churn)

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=avg_balance, x="NumOfProducts", y="Balance",
                 hue="NumOfProducts", palette=colors, legend=False)

# Value labels on every bar
for i, v in enumerate(avg_balance["Balance"]):
    ax.text(i, v + 1000, f'{v:,.0f}', ha='center', fontweight='bold')

plt.title('Average balance by Number of Products', fontsize=14)
plt.xlabel('Number of Products', fontsize=12)
plt.ylabel('Average balance ($)', fontsize=12)
plt.tight_layout()
plt.show()

## Finding: the loyal segment is not the valuable segment

Two-product customers hold about **$52k** and churn at **7.6%**. One-product customers hold about **$99k** and churn at **27.7%**. Four-product customers hold about **$94k** and all of them left.

**Decision**

- Do not read "low balance" as "safe to ignore," and do not read "many products" as "safe to ignore."
- Move high-balance **one-product** customers toward a **second** product. That is the only step the data supports. A third product is the step to refuse.
- Report impact as **balance at risk** (exits × average balance), not as a churn percentage alone. Percentages made the 60-person group look like the crisis. Dollars show the crisis is the one-product book.

In [ ]:

plt.figure(figsize=(8, 5))
sns.kdeplot(data=df, x="CreditScore", hue="Exited", common_norm=False,
            fill=True, alpha=0.3, palette={0: '#3498db', 1: '#e74c3c'})

plt.title('Credit Score Distribution: Stayed vs Exited', fontsize=14)
plt.xlabel('Credit Score', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.legend(title='Customer Status', labels=['Exited (1)', 'Stayed (0)'])
plt.tight_layout()
plt.show()

## Finding: credit score does not earn a place on the contact list

The distributions for stayers and leavers sit on top of each other. Both peak around **620–680** and both run from about **350 to 850**. Average scores are **652** (stayed) and **645** (left).

**Decision:** do not target retention by credit band. People leave at every score, which points to product fit, price, and engagement — the factors already measured — rather than credit stress. A slightly heavier tail below 450 is too small to staff a programme on.

In [ ]:
pd.pivot_table(data=df, index="Exited", columns="Tenure",
               values="Balance", aggfunc="mean").round(1)

## Finding: high balances leave at every year of tenure

At every tenure from 0 to 10 years, people who left held more money than people who stayed. Stayers sit near **$70–75k**. Leavers sit near **$84–98k**. The gap is about **$12–27k**, including customers who had already been with the bank for 8–9 years.

**Decision**

- Do not wait for "new customers" or "anniversary year 1." The balance gap is there on day one and in year ten.
- Keep balance in the score, next to product count and activity.
- When the retention team asks "why call this person," the answer is the deposit, not the length of the relationship.
- The cause (rate, fees, a competitor) is not in this table. Exit calls on high-balance leavers are how that cause gets learned. The table only decides **who** those calls are for.

In [ ]:
# Churn rate by salary decile (larger groups, more reliable)
df["SalaryBand"] = pd.qcut(df["EstimatedSalary"], 10, labels=False) + 1
print((df.groupby("SalaryBand")["Exited"].mean() * 100).round(1))

# Significance test for the top 30
from scipy.stats import binomtest
print(binomtest(3, 30, p=0.204).pvalue)

# Salary vs churn over the full data
from scipy.stats import mannwhitneyu
print(mannwhitneyu(df[df.Exited==1]["EstimatedSalary"],
                   df[df.Exited==0]["EstimatedSalary"]).pvalue)

## Finding: salary is flat, so it stays off the list

Churn across ten salary bands runs **19.3% to 21.9%**, around the book rate of **20.4%**. A Mann–Whitney test finds no salary difference between stayers and leavers (**p = 0.23**). The top 30 earners look safer (10% churn) only because the sample is tiny (**p = 0.25**).

**Decision:** do not build income-tier retention. Salary does not change the probability enough to justify a different offer.

In [ ]:
import matplotlib.pyplot as plt

df["AgeGroup"] = df["Age"].apply(lambda x: "Over 41" if x > 41 else "younger")
churn_rate = (df.groupby("AgeGroup")["Exited"].mean() * 100).round(1)
print(churn_rate)

plt.figure(figsize=(6, 5))
ax = churn_rate.loc[["younger", "Over 41"]].plot(kind="bar", color=["#3498db", "#e74c3c"])
for i, v in enumerate(churn_rate.loc[["younger", "Over 41"]]):
    ax.text(i, v + 0.5, f"{v}%", ha="center", fontweight="bold")

plt.axhline(df["Exited"].mean() * 100, color="gray", linestyle="--", label="Overall churn rate")
plt.title("Churn Rate by Age Group")
plt.ylabel("Churn Rate (%)")
plt.xlabel("")
plt.xticks(rotation=0)
plt.ylim(0, 50)
plt.legend()
plt.tight_layout()
plt.show()

## Finding: over 41 is where the book actually leaks

Customers **older than 41** churn at **39.9%**. Customers **41 or younger** churn at **11.1%**. That is about **3.6×**, a **29 point** gap — much larger than gender (8.6 points), and far larger than credit score, salary, or tenure.

They are a minority of customers and a large share of exits. Keeping this band changes the bank's total churn. Keeping the young band does not.

**Decision**

1. Give the over-41 book a named owner: relationship manager, a reviewed savings or term rate, and service that does not assume the customer wants an app-only relationship.
2. Stack age with **balance** and **product count** before spending. An older customer with a high balance costs more to lose than an older customer with an empty account.
3. Use age as a **priority feature** in the model and as a monthly management report. Do not use it as a slogan. The customer cannot get younger; the bank can decide who gets the call.

In [ ]:
young = df["Age"] <= 41          # boolean Series (mask)


zero_balance = df["Balance"] ==0
print(df.loc[zero_balance,"Exited"].value_counts(normalize=True).mul(100).round(1))
print(df.loc[~zero_balance,"Exited"].value_counts(normalize=True).mul(100).round(1))
young_zero = young & zero_balance
print(f"Customers: {young_zero.sum():,}")
print(df.loc[young_zero, "Exited"].value_counts(normalize=True).mul(100).round(1))

young_more = young & ~zero_balance

print(f"Customers: {young_more.sum():,}")
print(df.loc[young_more, "Exited"].value_counts(normalize=True).mul(100).round(1))

## Finding: empty accounts are the stable ones

Zero-balance customers churn at **13.8%**. Customers holding a balance churn at **24.1%** (about **1.7×**). Inside the younger group the same pattern holds: **7.1%** with no balance versus **13.5%** with a balance. So this is not just age in disguise.

Young, zero-balance customers (2,547 people) are the safest pocket in the book, at about a third of the overall rate. They are also the pocket with nothing to lose.

**Decision**

- Spend retention effort on customers **who hold a balance**, and first on those over 41, where the two risks stack.
- Do not buy a campaign for young zero-balance customers. They are already the stable group.
- Benefits should protect the deposit (rate, a named contact), because that is the asset walking out the door.

In [ ]:
old = df["Age"] > 41   # added: this variable was not defined in the original

import numpy as np


has_balance = df["Balance"] > 0

seg = (df.assign(AgeGroup=np.where(old, "Over 41", "41 or younger"),
                 BalanceGroup=np.where(has_balance, "Balance > 0", "Zero balance"))
         .groupby(["Gender", "AgeGroup", "BalanceGroup"])["Exited"]
         .agg(Customers="size", Churned="sum", ChurnRate="mean"))
seg["ChurnRate"] = (seg["ChurnRate"] * 100).round(1)
print(seg)

# Share of all churners coming from the red-alert segment
red = (df["Gender"] == "Female") & old & has_balance
print(f"Segment size: {red.sum():,} ({red.mean()*100:.1f}% of customers)")
print(f"Churn rate: {df.loc[red, 'Exited'].mean()*100:.1f}%")
print(f"Share of all churners: {df.loc[red, 'Exited'].sum() / df['Exited'].sum() * 100:.1f}%")

## Finding: the sharpest pocket is older women who hold a balance

Women over 41 with a balance churn at **51.8%**. That is **2.5×** the book rate (20.4%) and above men in the same age-and-balance group (**38.8%**).

This pocket is **1,012 customers (10% of the book)** and **524 exits (26% of all churners)**. The segment holds about **$121 million** in deposits. Risk runs from **5.5%** (young men, zero balance) to **51.8%** here — almost a **10×** range — so a single bank-wide treatment is the wrong product.

**How I would use this without turning gender into an offer rule**

1. The **operating list** is everyone **over 41 with a balance**: 2,145 customers, about **45%** churn, and about **$116 million** of deposits already in the exits from this wider group. Men in that group are high risk too. Age and balance are the fields a retention policy can apply evenly.
2. The gender split is the **diagnostic**. Within every age and balance group, women leave more often. That justifies a service review (rates, branch experience, product fit), not a different price for women.
3. Track three numbers monthly on the over-41-with-balance list: churn rate, balance retained, and cost per customer kept, against a holdout group that does not get the offer. If the offer does not beat the holdout, stop it.

## 11. What EDA changed in the model

| Feature | What the bank should conclude | Strength |
|---|---|---|
| Age | Over 41 is the leak; use it to prioritise | Strongest |
| IsActiveMember | Inactive customers are 2× and are a campaign the bank can run | Strongest |
| Geography | Germany is double the rate and double the balance | Strong |
| NumOfProducts | 2 is safe; 3–4 is a broken cross-sell | Strong |
| Balance | Higher balances leave; empty accounts are safer | Strong |
| Tenure | Flat; do not fund an onboarding rescue on this alone | Weak |
| Gender | Real gap; diagnose it, do not price on it | Moderate |
| Credit score, salary, credit card | No separation; leave them off the targeting rules | None |

Everything below is a consequence of this table. Features are added only where the pattern is non-linear or combined. Fields that do not move churn are not asked to carry the model.